# Spark Key Concepts

1) **Spark Architecture**  
   Spark is a distributed computing framework that processes large-scale data across clusters. It consists of a driver program, cluster manager, and worker nodes.

2) **Master Node / Driver Node / Header Node / Name Node**  
   The master node (also called driver node) is responsible for orchestrating the execution of Spark applications. It maintains information about the cluster, schedules jobs, and coordinates tasks. In Hadoop, the Name Node manages metadata, but in Spark, the driver handles job scheduling and resource allocation.

3) **Worker Node / Data Node / Slave Node / Process Node**  
   Worker nodes (also called slave nodes) execute tasks assigned by the master. They store data and run computations. In Hadoop, Data Nodes store actual data blocks, while in Spark, worker nodes process data partitions.

4) **Executor**  
   An executor is a JVM process launched on each worker node. It is allocated a specific amount of RAM and CPU (vCPU/slots). Executors run tasks and keep data in memory for fast processing.

5) **Context (SparkContext, SQLContext)**  
   Context objects are entry points for Spark functionality. `SparkContext` is the main entry for core Spark features, while `SQLContext` provides access to Spark SQL capabilities. They define the working area and resources allocated for specific operations.

6) **SparkSession**  
   `SparkSession` is the unified entry point for Spark applications. It handles authentication (establishing a session with the cluster) and authorization (managing user privileges). It combines `SparkContext` and `SQLContext` into a single object.

7) **Spark Job Internal Process**  
   The execution flow in Spark:  
   - **Code**: User writes Spark code.  
   - **Job**: The code is converted into one or more jobs.  
   - **Stages**: Each job is split into stages based on data shuffling.  
   - **Tasks**: Stages are further divided into tasks, which are the smallest units of work executed by executors on partitions.
8) Partitions (spark devides data into logical partitions to process data on multiple worker nodes)-- logical partitions with default 128MB Size

#Pyspark


## 1) What is a DataFrame?

A **DataFrame** in Spark is a distributed collection of data organized into named columns, similar to a table in a relational database or a data frame in Python's pandas library. DataFrames provide a higher-level abstraction for working with structured and semi-structured data, enabling efficient querying, transformation, and analysis at scale.

---

## 2) How to Create a DataFrame?

Spark provides multiple APIs to create DataFrames:

- **Reading from files**:  
  Use `spark.read` to load data from various formats:
  - CSV: `spark.read.csv("path")`
  - JSON: `spark.read.json("path")`
  - Parquet: `spark.read.parquet("path")`
  - ORC, Avro, Delta: `spark.read.format("format").load("path")`
- **From SQL queries**:  
  `spark.sql("SELECT * FROM table")`
- **From Python objects**:  
  `spark.createDataFrame(data, schema)`

These methods allow you to ingest data from external sources or create DataFrames from in-memory data.

---

## 3) Types of Spark Operations

Spark operations on DataFrames are categorized as:

- **Transformations**:  
  Operations that produce a new DataFrame from an existing one (e.g., `select`, `filter`). Transformations are *lazy*; they are not executed until an action is called.
- **Actions**:  
  Operations that trigger computation and return results (e.g., `show`, `collect`, `write`). Actions cause Spark to execute the transformations and return data to the driver or write it to storage.

---

## 4) Actions in Spark

Actions are operations that return a value to the driver program or write data externally. Common actions include:

- **Displaying data**:  
  - `df.show()`: Prints rows in tabular format.
  - `df.display()`: Enhanced display in Databricks notebooks.
- **Collecting data**:  
  - `df.collect()`: Returns all rows as a list to the driver.
- **Writing data**:  
  - `df.write.format("format").save("path")`: Saves DataFrame to storage.
  - `df.write.format("format").saveAsTable("table")`: Saves DataFrame as a managed table.

All write operations are considered actions because they trigger Spark jobs.

---

## 5) Types of Transformations

Transformations modify DataFrames and are classified as:

### a) Metadata Transformations

- **Add column**: `df.withColumn("new_col", expr)`
- **Change data type**: `df.withColumn("col", df.col.cast("type"))`
- **Rename column**: `df.withColumnRenamed("old", "new")`, `df.toDF("col1", "col2")`
- **Drop column(s)**: `df.drop("col")`
- **Select columns**: `df.select("col1", "col2")`
- **Select with expressions**: `df.selectExpr("col1 as new_col", "col2 * 2")`

### b) Data Transformations

- **Remove duplicates**: `df.distinct()`, `df.dropDuplicates()`
- **Remove nulls**: `df.dropna()`, `df.na.drop()`
- **Fill nulls**: `df.fillna(value)`, `df.na.fill(value)`
- **Filter rows**: `df.filter(condition)`, `df.where(condition)`
- **Group and aggregate**:  
  - `df.groupBy("col").agg({"col2": "max"})`
  - Aggregations: `min`, `max`, `avg`, `count`, `sum`, `stdev`
- **Sort data**:  
  - `df.orderBy("col")`, `df.sort("col")`
  - Specify order: `.asc()`, `.desc()`
- **Set operations**:  
  - Merge DataFrames: `df.union(df1)`, `df.unionAll(df1)`
  - Subtract: `df.minus(df1)`
  - Intersect: `df.intersect(df1)`
- **Join operations**:  
  - `df.join(df1, condition, "type")` (types: inner, left, right, full, semi, anti)
  - `df.crossJoin(df2)`
- **Partitioning**:  
  - Increase/decrease: `df.repartition(n)`
  - Decrease only: `df.coalesce(n)`

---

## 6) Narrow vs Wide Transformations

- **Narrow Transformations**:  
  Each input partition contributes to only one output partition. No data shuffling occurs. Examples:
  - `df.withColumn()`
  - `df.withColumnRenamed()`
  - `df.drop()`
  - `df.select()`
  - `df.selectExpr()`
  - `df.filter()`, `df.where()`
  - `df.union(df1)`
  - `df.coalesce(n)`

- **Wide Transformations**:  
  Data from multiple input partitions may be needed for one output partition, causing a shuffle across the cluster. Examples:
  - `df.distinct()`, `df.dropDuplicates()`
  - `df.groupBy().agg()`
  - `df.orderBy()`, `df.sort()`, `df.sortWithinPartitions()`
  - `df.join(df1)`
  - `df.crossJoin(df1)`
  - `df.minus(df1)`, `df.intersect(df1)`
  - `df.repartition(n)`

**Note:**  
- *Narrow transformations* are more efficient as they avoid shuffling.
- *Wide transformations* involve shuffling, which can impact performance due to network and disk I/O.

---

# Delta Lake


# Delta Lake: Detailed Documentation

## 1) What is Delta Lake?

Delta Lake is an open-source storage layer that brings reliability, scalability, and performance to data lakes. It enables ACID transactions, scalable metadata handling, and unifies streaming and batch data processing. Delta Lake is built on top of Apache Spark and stores data in Parquet format, while maintaining a transaction log for versioning and consistency.

---

## 2) Delta Lake Architecture

Delta Lake stores data in Parquet files within a directory. Alongside the data, it maintains a special folder called `_delta_log` that contains metadata, transaction logs, indexes, and statistics in JSON and CRC files. This architecture enables features like ACID transactions, schema enforcement, and time travel.

- **Data Storage**: Data is stored as Parquet files for efficient columnar storage and compression.
- **Transaction Log**: The `_delta_log` directory contains JSON files that record every change (add, remove, update) to the table, ensuring atomicity and consistency.
- **Metadata & Stats**: The log files also store schema information, statistics, and indexes to optimize query performance and enable features like time travel.

---

## 3) Delta Lake Features

### 1. ACID Transactions

Delta Lake supports atomic, consistent, isolated, and durable (ACID) transactions. This ensures that all writes are either fully completed or not applied at all, preventing partial or corrupt data.

### 2. Schema Enforcement

Delta Lake enforces the schema of the table during write operations. If incoming data does not match the table schema, the write will fail unless schema evolution is enabled.

- **mergeSchema=False**: Strict schema enforcement; mismatched columns cause errors.

### 3. Schema Evolution

Delta Lake allows the schema of a table to evolve over time. New columns can be added, and data types can be changed as needed.

- **mergeSchema=True**: Automatically merges new columns into the table schema.
- **overwriteSchema=True**: Overwrites the existing schema with the new schema.

### 4. Time Travel

Delta Lake maintains a history of all changes, allowing users to query previous versions of the data. This is useful for auditing, debugging, and reproducing experiments.

- **Syntax**: `SELECT * FROM table VERSION AS OF <version_number>`

### 5. Audit Logs

All changes to Delta tables are recorded in the transaction log, providing a complete audit trail of data modifications.

### 6. Merge (Upsert Operations)

Delta Lake supports the `MERGE` operation, which allows incremental data loading by upserting (inserting or updating) records based on specified conditions.

- **Use Case**: Efficiently handle slowly changing dimensions and CDC (Change Data Capture).

### 7. Partitioning

Delta tables can be partitioned by one or more columns to improve query performance and manageability.

- **Syntax**: `PARTITIONED BY (column_name)`

### 8. Liquid Clustering (Cluster By)

Delta Lake supports clustering data by columns to optimize data layout and query performance. This is more flexible than traditional partitioning.

- **Syntax**: `CLUSTER BY (column_name)`

### 9. Vacuum (Purging Old Snapshots)

The `VACUUM` command removes obsolete files and data that are no longer referenced by the transaction log, freeing up storage and maintaining table health.

- **Syntax**: `VACUUM table_name RETAIN <hours>`

### 10. Optimize (Compacting Small Files)

The `OPTIMIZE` command compacts small files into larger ones, improving read performance and reducing metadata overhead.

- **Syntax**: `OPTIMIZE table_name`

### 11. Optimize with ZORDER (Data Skipping)

`OPTIMIZE ... ZORDER BY` physically sorts data files by specified columns, enabling efficient data skipping and faster queries.

- **Syntax**: `OPTIMIZE table_name ZORDER BY (column_name)`

---

Delta Lake combines the reliability of data warehouses with the scalability of data lakes, making it a powerful solution for modern data engineering and analytics.

#ADF


# Azure Data Factory (ADF) Overview

Azure Data Factory (ADF) is a cloud-based data integration service that enables you to create, schedule, and orchestrate data pipelines for moving and transforming data from various sources to destinations.

---

## 1) Why Use ADF?

- **Ingestion and Orchestration:**  
  ADF is designed for ingesting data from diverse sources and orchestrating complex data workflows. It automates data movement and transformation across cloud and on-premises environments.

- **External System Data Migration:**  
  For migrating data from external systems (e.g., on-premises databases, SaaS applications) to cloud storage or data lakes, ADF acts as a robust ingestion tool.

---

## 2) Integration Runtime (IR)

- **Self-Hosted Integration Runtime:**  
  A self-hosted IR is an ADF component installed on-premises or in a private network. It enables secure data movement and transformation between on-premises sources and cloud destinations, overcoming network boundaries and firewalls.

---

## 3) Linked Service

- **Definition:**  
  A linked service defines the connection information required for ADF to connect to external resources.  
- **Examples:**  
  - Databases (SQL Server, Oracle, MySQL, etc.)
  - File systems (Azure Data Lake, Blob Storage)
  - Cloud services (Databricks, Key Vault)
  - External APIs and other cloud systems

Linked services act as connection managers, storing credentials and endpoints securely.

---

## 4) Pipeline

- **Definition:**  
  A pipeline is a logical grouping of activities that perform data movement and transformation.  
- **Purpose:**  
  Pipelines enable workflow orchestration, allowing you to chain activities, implement control flow (loops, conditions), and manage dependencies.

- **Typical Activities in a Pipeline:**  
  - Data copying
  - Data transformation (using Databricks, SQL, etc.)
  - Data validation and metadata extraction
  - Orchestration logic (looping, conditional branching)

---

## 5) Activities in ADF

Activities are the building blocks of pipelines. Each activity performs a specific operation:

- **Copy Activity:**  
  Moves data from a source to a destination (data migration).

- **Get Metadata Activity:**  
  Retrieves metadata (e.g., schema, file size) from data sources.

- **Lookup Activity:**  
  Executes queries or reads data for validation or extraction.

- **Filter Activity:**  
  Filters data or pipeline items based on conditions.

- **ForEach Activity:**  
  Iterates over a collection of items, executing child activities for each item.

- **If Condition Activity:**  
  Implements branching logic based on evaluated conditions.

- **Switch Activity:**  
  Routes execution to different branches based on matching values.

- **Execute Pipeline Activity:**  
  Invokes another pipeline from within a pipeline, enabling modular workflows.

- **Wait Activity:**  
  Pauses pipeline execution for a specified duration.

- **Stored Procedure Activity:**  
  Executes stored procedures in supported databases.

- **Web Activity:**  
  Integrates with REST APIs or web services for data retrieval or triggering external processes.

- **Databricks Notebook Activity:**  
  Executes Databricks notebooks for advanced data processing and analytics.

- **Databricks Job Activity:**  
  Triggers Databricks jobs or pipelines for scalable data engineering tasks.

---

## Summary

ADF provides a scalable, flexible, and secure platform for building end-to-end data integration solutions. Its modular architecture (linked services, pipelines, activities, integration runtimes) supports a wide range of data movement, transformation, and orchestration scenarios across hybrid environments.

#python

1) **Variables**  
   Variables are used to store data in Python. You assign a value to a variable using the `=` operator.  
   Example:  
   python
   x = 10
   name = "Alice"
   
   Variables can hold different data types such as integers, floats, strings, lists, etc.

2) **Print Function / Format Function**  
   The `print()` function outputs data to the console.  
   Example:  
   python
   print("Hello, World!")
   
   For formatted output, use f-strings or the `format()` method:  
   python
   print(f"Name: {name}, Age: {x}")
   print("Name: {}, Age: {}".format(name, x))
   

3) **Loops (for loop, while loop)**  
   Loops are used to repeat actions.  
   - **For loop**: Iterates over a sequence (list, tuple, string, etc.)  
     python
     for i in range(5):
         print(i)
     
   - **While loop**: Repeats as long as a condition is true  
     python
     count = 0
     while count < 5:
         print(count)
         count += 1
     

4) **Conditions (if, else, elif)**  
   Conditional statements control the flow of execution based on conditions.  
   Example:  
   python
   if x > 5:
       print("x is greater than 5")
   elif x == 5:
       print("x is equal to 5")
   else:
       print("x is less than 5")
   

5) **Collections (list, tuple, set, dict)**  
   Python provides several built-in collection types:  
   - **List**: Ordered, mutable sequence  
     python
     my_list = [1, 2, 3]
     
   - **Tuple**: Ordered, immutable sequence  
     python
     my_tuple = (1, 2, 3)
     
   - **Set**: Unordered, unique elements  
     python
     my_set = {1, 2, 3}
     
   - **Dictionary**: Key-value pairs  
     python
     my_dict = {"name": "Alice", "age": 25}
     

6) **Functions**  
   Functions are reusable blocks of code that perform a specific task.  
   Define a function using `def`:  
   python
   def greet():
       print("Hello!")
   greet()
   

7) **Functions with Parameters**  
   Functions can accept parameters to work with different data.  
   Example:  
   python
   def add(a, b):
       return a + b
   result = add(5, 3)
   print(result)
   

8) **Exception Handling**  
   Exception handling allows you to manage errors gracefully using `try`, `except`, `finally`.  
   Example:  
   python
   try:
       value = int(input("Enter a number: "))
   except ValueError:
       print("Invalid input! Please enter a number.")
   finally:
       print("Execution completed.")

#SQL

## 1) SELECT Statement

The `SELECT` statement is used to query data from one or more tables or views. It supports various clauses and operations:
- **Joins**: Combine rows from two or more tables based on related columns (e.g., `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, `FULL JOIN`).
- **GROUP BY**: Aggregate data across rows sharing the same values in specified columns (e.g., `SUM`, `COUNT`, `AVG`).
- **ORDER BY**: Sort the result set by one or more columns, in ascending (`ASC`) or descending (`DESC`) order.
- **Set Operators**: Combine results from multiple queries (e.g., `UNION`, `INTERSECT`, `EXCEPT`).
- **CASE**: Conditional logic within queries to return values based on conditions.
- **LIMIT**: Restrict the number of rows returned by the query.

**Example:**
sql
SELECT department, COUNT(*) AS num_employees
FROM employees
WHERE salary > 50000
GROUP BY department
ORDER BY num_employees DESC
LIMIT 10;


---

## 2) DML Operations: INSERT, UPDATE, DELETE, MERGE

- **INSERT**: Add new rows to a table.
  sql
  INSERT INTO employees (id, name, department) VALUES (1, 'Alice', 'HR');
  
- **UPDATE**: Modify existing rows in a table.
  sql
  UPDATE employees SET salary = salary * 1.1 WHERE department = 'Sales';
  
- **DELETE**: Remove rows from a table.
  sql
  DELETE FROM employees WHERE id = 1;
  
- **MERGE**: Perform upsert operations (insert, update, or delete) based on matching conditions.
  sql
  MERGE INTO target_table t
  USING source_table s
  ON t.id = s.id
  WHEN MATCHED THEN UPDATE SET t.value = s.value
  WHEN NOT MATCHED THEN INSERT (id, value) VALUES (s.id, s.value);
  

---

## 3) DDL Operations: CREATE, ALTER, DROP, TRUNCATE

- **CREATE**: Define new tables, views, or other database objects.
  sql
  CREATE TABLE employees (id INT, name STRING, department STRING);
  
- **ALTER**: Modify the structure of existing objects (e.g., add/drop columns).
  sql
  ALTER TABLE employees ADD COLUMN hire_date DATE;
  
- **DROP**: Remove database objects permanently.
  sql
  DROP TABLE employees;
  
- **TRUNCATE**: Remove all rows from a table without deleting the table itself.
  sql
  TRUNCATE TABLE employees;
  

---

## 4) Metadata and Information Commands

- **DESCRIBE**: Show the schema of a table or view.
  sql
  DESCRIBE employees;
  
- **DESCRIBE HISTORY**: View the audit log/history of a Delta table (track changes over time).
  sql
  DESCRIBE HISTORY employees;
  
- **DESCRIBE DETAIL**: Show detailed properties and statistics of a table.
  sql
  DESCRIBE DETAIL employees;
  
- **DESCRIBE EXTENDED**: Display extended metadata, including table properties.
  sql
  DESCRIBE EXTENDED employees;
  
- **SHOW CREATE TABLE**: Output the DDL statement used to create a table.
  sql
  SHOW CREATE TABLE employees;
  
- **SHOW TABLES**: List all tables in the current or specified database.
  sql
  SHOW TABLES;
  
- **SHOW DATABASES**: List all databases in the system.
  sql
  SHOW DATABASES;
  
- **SHOW VIEWS**: List all views in the current or specified database.
  sql
  SHOW VIEWS;
  

---

## 5) EXPLAIN Statement

The `EXPLAIN` command displays the execution plan for a SQL query, showing how Spark will execute the query, including stages, partitions, and operations. This helps in understanding and optimizing query performance.

**Example:**
sql
EXPLAIN SELECT * FROM employees WHERE department = 'IT';

#databricks


# Databricks Key Concepts and Integration: Detailed Documentation

---

## 1) How to Call One Notebook from Another

- **%run Command**:  
  Use `%run ./notebook_path` at the top of a notebook to include and execute another notebook's code inline. All variables and functions become available in the current notebook's scope.
  - *Example*:  
    
    %run ./SharedUtilities
    
  - *Use Case*: Sharing reusable code, functions, or setup logic.

- **dbutils.notebook.run()**:  
  Programmatically runs another notebook as a separate job, optionally passing parameters and capturing its output.
  - *Example*:  
    python
    result = dbutils.notebook.run("notebook_path", timeout_seconds=60, arguments={"param1": "value"})
    
  - *Use Case*: Modular workflows, chaining notebooks, parameterization.

---

## 2) Difference Between %run and dbutils.notebook.run

- **%run**:  
  - Executes the target notebook inline; variables/functions are imported into the current notebook.
  - No parameter passing or output retrieval.
  - Synchronous and shares the same Spark context.

- **dbutils.notebook.run**:  
  - Runs the target notebook as a separate job/subprocess.
  - Supports parameter passing and output retrieval.
  - Isolated execution context; does not share variables/functions.
  - Returns output via `dbutils.notebook.exit()`.

---

## 3) How to Pass Parameters to a Notebook

- **Widgets**:  
  Use Databricks widgets to define parameters that can be set when running a notebook.
  - *Create a text widget*:  
    python
    dbutils.widgets.text("param_name", "default_value", "Description")
    
  - *Pass parameters via dbutils.notebook.run*:  
    python
    dbutils.notebook.run("notebook_path", 60, {"param_name": "value"})
    

---

## 4) How to Retrieve Parameters in a Notebook

- **Get Widget Value**:  
  Use `dbutils.widgets.get("param_name")` to access the value of a widget/parameter.
  - *Example*:  
    python
    param_value = dbutils.widgets.get("param_name")
    

---

## 5) Cluster Types in Databricks

- **All-Purpose Compute**:  
  Interactive clusters for notebooks, ad-hoc analysis, and collaborative development.

- **Job Compute**:  
  Dedicated clusters for running scheduled jobs and production workloads.

- **Serverless Compute**:  
  Managed, auto-scaling clusters for optimized resource usage and cost efficiency.

---

## 6) Job Types in Databricks

- **Spark Job**:  
  Executes Spark code (Scala, Python, SQL) for distributed data processing.

- **Notebook Job**:  
  Runs a Databricks notebook as a scheduled or triggered job.

- **Python Job**:  
  Executes standalone Python scripts.

- **SQL Job**:  
  Runs SQL queries or scripts.

- **Spark Submit Job**:  
  Submits Spark applications using the `spark-submit` interface.

---

## 7) How to Retrieve Notebook Output

- **dbutils.notebook.exit(msg)**:  
  Use this function to return a value or message from a notebook when called via `dbutils.notebook.run`.
  - *Example*:  
    python
    dbutils.notebook.exit("Success")
    
  - The returned value is captured in the calling notebook.

---

## 8) Integrating Databricks Notebooks in Azure Data Factory (ADF)

- **ADF Linked Service**:  
  Connects ADF to Databricks workspace using token-based authentication.

- **Notebook Activity**:  
  Executes Databricks notebooks within ADF pipelines for data transformation or analytics.

- *Steps*:  
  1. Create a Databricks linked service in ADF.
  2. Add a Notebook activity to your pipeline.
  3. Configure parameters and authentication (token).

---

## 9) Storing Credentials in Azure

- **Azure Key Vault**:  
  Securely stores secrets, tokens, passwords, and keys.  
  - *Use Case*: Centralized credential management for cloud resources.

---

## 10) Retrieving Credentials from Azure

- **ADF Linked Service to Key Vault**:  
  ADF can access secrets from Key Vault via linked service integration.
  - *Example*:  
    Configure your pipeline to reference secrets for authentication.

---

## 11) Retrieving Credentials from Databricks

- **Databricks Secrets**:  
  Store and retrieve secrets using Databricks Secret Scopes.
  - *Retrieve a secret*:  
    python
    token = dbutils.secrets.get(scope="scope_name", key="token_key")
    

---

## 12) Integrating Data Lake with Databricks

- **Unity Catalog External Location**:  
  Register external storage (e.g., Azure Data Lake) in Unity Catalog for secure, managed access.
  - *Use Case*: Read/write data from/to data lakes using Databricks tables.

---

## 13) What is a Volume?

- **Volume**:  
  A storage object in Databricks for storing files and data, backed by the workspace's default storage account.
  - *Features*:  
    - Access/security controls via Unity Catalog.
    - Part of a schema; supports managed access.

---

## 14) What is Unity Catalog?

- **Unity Catalog**:  
  Centralized metadata management for Databricks, supporting region-level sharing across multiple workspaces.
  - *Hierarchy*:  
    - **Catalogs** → **Schemas** → **Tables/Views/Volumes**
  - *Benefits*:  
    - Fine-grained access control, data governance, and auditability.

---

## 15) How to Retrieve Data from a Volume

- **Path-Based Access**:  
  Access files in a volume using its path.
  - *Example*:  
    python
    df = spark.read.csv("/Volumes/catalog/schema/volume/file.csv")
    

---

## 16) Types of Views in Databricks

- **Temporary View**:  
  Session-scoped; not persisted.

- **Global Temporary View**:  
  Available across all sessions; stored in a reserved database (`global_temp`).

- **Permanent View**:  
  Persisted in the metastore; available until explicitly dropped.

---

## 17) Types of Tables in Databricks

- **Managed Table**:  
  Databricks manages both data and metadata; data stored in workspace storage.

- **External Table**:  
  Metadata managed by Databricks; data stored externally (e.g., in a data lake).

---

In [0]:
# Metadata Transformations
metadata_transformations = [
    "df.withColumn()",            # Add or modify column
    "df.withColumnRenamed()",     # Rename column
    "df.toDF()",                  # Rename all columns
    "df.drop()",                  # Drop column(s)
    "df.select()",                # Select columns
    "df.selectExpr()"             # Select columns with expressions
]

# Data Transformations
data_transformations = [
    "df.distinct()",              # Remove duplicates
    "df.dropDuplicates()",        # Remove duplicates
    "df.dropna()",                # Remove null rows
    "df.na.drop()",               # Remove null rows
    "df.fillna()",                # Fill nulls
    "df.na.fill()",               # Fill nulls
    "df.filter()",                # Filter rows
    "df.where()",                 # Filter rows
    "df.groupBy()",               # Group data
    "df.groupBy().agg()",         # Aggregate data
    "df.orderBy()",               # Sort data
    "df.sort()",                  # Sort data
    "df.union(df1)",              # Merge dataframes (set operator)
    "df.unionAll(df1)",           # Merge dataframes (set operator)
    "df.minus(df1)",              # Set difference
    "df.intersect(df1)",          # Set intersection
    "df.join(df1, condition, type)", # Join dataframes
    "df.crossJoin(df2)",          # Cross join
    "df.repartition(n)",          # Change partitions
    "df.coalesce(n)"              # Decrease partitions
]

# Narrow Transformations (no shuffle)
narrow_transformations = [
    "df.withColumn()",
    "df.withColumnRenamed()",
    "df.drop()",
    "df.select()",
    "df.selectExpr()",
    "df.where()",
    "df.filter()",
    "df.union(df1)",
    "df.unionAll(df1)",
    "df.coalesce(n)"
]

# Wide Transformations (with shuffle)
wide_transformations = [
    "df.distinct()",
    "df.dropDuplicates()",
    "df.groupBy().agg()",
    "df.orderBy()",
    "df.sort()",
    "df.sortWithinPartitions()",
    "df.join(df1, condition, type)",
    "df.crossJoin(df1)",
    "df.minus(df1)",
    "df.intersect(df1)",
    "df.repartition(n)"
]

display([
    {"Concept": "Metadata", "Transformations": metadata_transformations},
    {"Concept": "Data", "Transformations": data_transformations},
    {"Concept": "Narrow", "Transformations": narrow_transformations},
    {"Concept": "Wide", "Transformations": wide_transformations}
])